In [1]:
!pip -q install -U \
langchain \
langchain-community \
langchain-google-genai \
langchain-groq \
langchain-chroma \
chromadb \
pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23

In [2]:
import os
import warnings
from getpass import getpass
warnings.filterwarnings("ignore")
from google.colab import files
from langchain_core.documents import Document #Represents a document in a structured format.
from langchain_text_splitters import RecursiveCharacterTextSplitter #Splits large documents into smaller chunks.

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from pypdf import PdfReader

In [4]:
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

Enter Groq API Key: ··········


In [13]:
uploaded = files.upload()
import pandas as pd
from langchain_core.documents import Document

df = pd.read_excel("/content/Project-Management-Sample-Data (1).xlsx")

# Convert rows into LangChain Document objects for your RAG pipeline
pages = [
    Document(page_content=" ".join(str(x) for x in row.values if pd.notna(x))) # Extract and join only non-null values from each row
    for _, row in df.iterrows()
]

print("Total Rows/Documents:", len(pages))

print(pages)

Saving Project-Management-Sample-Data.xlsx to Project-Management-Sample-Data (2).xlsx
Total Rows/Documents: 51
[Document(metadata={}, page_content='Excel Sample Data'), Document(metadata={}, page_content=''), Document(metadata={}, page_content='Project Management Data'), Document(metadata={}, page_content=''), Document(metadata={}, page_content='Project Name Task Name Assigned to Start Date Days Required End Date Progress'), Document(metadata={}, page_content='Marketing Market Research Alice 2024-01-01 00:00:00 13 2024-01-14 00:00:00 0.78'), Document(metadata={}, page_content='Marketing Content Creation Bob 2024-01-14 00:00:00 14 2024-01-28 00:00:00 1'), Document(metadata={}, page_content='Marketing Social Media Planning Charlie 2024-01-28 00:00:00 22 2024-02-19 00:00:00 0.45'), Document(metadata={}, page_content='Marketing Campaign Analysis Daisy 2024-02-18 00:00:00 25 2024-03-14 00:00:00 0'), Document(metadata={}, page_content='Product Dev Prototype Development Ethan 2024-01-02 00:00

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
chunks = splitter.split_documents(pages)
print(len(chunks))

print(chunks)

print(chunks[0])

print(chunks[0].page_content)


49
[Document(metadata={}, page_content='Excel Sample Data'), Document(metadata={}, page_content='Project Management Data'), Document(metadata={}, page_content='Project Name Task Name Assigned to Start Date Days Required End Date Progress'), Document(metadata={}, page_content='Marketing Market Research Alice 2024-01-01 00:00:00 13 2024-01-14 00:00:00 0.78'), Document(metadata={}, page_content='Marketing Content Creation Bob 2024-01-14 00:00:00 14 2024-01-28 00:00:00 1'), Document(metadata={}, page_content='Marketing Social Media Planning Charlie 2024-01-28 00:00:00 22 2024-02-19 00:00:00 0.45'), Document(metadata={}, page_content='Marketing Campaign Analysis Daisy 2024-02-18 00:00:00 25 2024-03-14 00:00:00 0'), Document(metadata={}, page_content='Product Dev Prototype Development Ethan 2024-01-02 00:00:00 18 2024-01-20 00:00:00 1'), Document(metadata={}, page_content='Product Dev Quality Assurance Fiona 2024-01-20 00:00:00 10 2024-01-30 00:00:00 0.78'), Document(metadata={}, page_conten

In [15]:
!pip install sentence-transformers langchain-huggingface

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
from langchain_chroma import Chroma
vector_db = Chroma.from_documents(documents=chunks, embedding=embedding, collection_name="genai_vectordb_rag_lab_class")

collection = vector_db._collection
count = collection.count()
print("Vectors Stored :", count)

Vectors Stored : 98


In [35]:
text = "what are the data in row 5"
vector = embedding.embed_query(text)

In [36]:
print(len(vector))
print(vector[:10])

384
[-0.01653890870511532, 0.027643674984574318, -0.03473931550979614, -0.013272409327328205, -0.0041998871602118015, 0.021346597000956535, -0.019608108326792717, -0.05312912538647652, -0.047283001244068146, 0.05721064656972885]


In [37]:
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k":2})

In [38]:
ans = retriever.invoke(text)
ans[0].page_content

'Excel Sample Data'

In [39]:
ans[1].page_content


'Excel Sample Data'